<a href="https://colab.research.google.com/github/Bassendiaye/mes_notebooks/blob/main/3_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importation des données

Nous importons pandas et utilisons la fonction read_csv pour lire les reviews sous format csv.

In [ ]:
import pandas as pd

imdb_file = 'IMDB Dataset.csv'
df=pd.read_csv(imdb_file)

In [ ]:
print(f'taille de mon data frame {df.shape}')
print(f'nom des colonnes {list(df.keys())}')

taille de mon data frame (50000, 2)
nom des colonnes ['review', 'sentiment']


Affichage des 5 premiers reviews

In [ ]:
print(df.head(5))

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


Affichage du review numero 4

In [ ]:
row=3
print(df['review'][row])
print(df['sentiment'][row])

Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going to make a film you must Decide if its a thriller or a drama! As a drama the movie is watchable. Parents are divorcing & arguing like in real life. And then we have Jake with his closet which totally ruins all the film! I expected to see a BOOGEYMAN similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. As for the shots with Jake: just ignore them.
negative


## Nettoyer les balises html

Tout d'abord installer le module '**beautifulsoup4**' si c'est pas déjà fait. Le mieux c'est de l'installer directement par Anaconda. Il faudra ensuite fermer et rouvrir jupyter notebook.

On écrit ensuite une fonction pour supprimer les balises html des données textuelles.

In [ ]:
from bs4 import BeautifulSoup

def remove_html_tags(raw_text):
    text = BeautifulSoup(raw_text).get_text()
    return text

Nous appliquons notre fonction sur le review numero 4 pour tester si tout fonctionne comme prévu.

In [ ]:
clean_text=remove_html_tags(df['review'][row])
print("AVANT NETTOYAGE:")
print(df['review'][row])
print("APRES NETTOYAGE:")
print(clean_text)

AVANT NETTOYAGE:
Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going to make a film you must Decide if its a thriller or a drama! As a drama the movie is watchable. Parents are divorcing & arguing like in real life. And then we have Jake with his closet which totally ruins all the film! I expected to see a BOOGEYMAN similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. As for the shots with Jake: just ignore them.
APRES NETTOYAGE:
Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo a

La sortie semble être satisfaisante. Maintenant, nous pouvons l'appliquer sur tout le corpus.

In [ ]:
df["review"]=df["review"].apply(lambda x: remove_html_tags(x))

/Users/bamba/opt/anaconda3/envs/tf/lib/python3.7/site-packages/bs4/__init__.py:439: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  MarkupResemblesLocatorWarning


In [ ]:
df.head(5)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


Nous utilisons un mappage pour coder les valeurs de sentiment: positive -> 1; negative ->0

In [ ]:
sentiment_map={'positive':1, 'negative':0}
print(sentiment_map['positive'])
print(sentiment_map['negative'])

1
0


Nous appliquons le mappage sur les valeurs de la colonne 'sentiment' de notre dataframe.

In [ ]:
df["sentiment"]=df["sentiment"].apply(lambda x:sentiment_map[x])
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. The filming tec...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


# Pré-traitement

In [ ]:
import spacy
from spacy.lang.en import English

In [ ]:
nlp = English()

Nous importons la liste des mots vides et créons une fonction pour les identifier.

In [ ]:
en_stopwords = spacy.lang.en.stop_words.STOP_WORDS

# Fonction pour détecter les mots vides
def is_stop_word(word, stop_list):
    return str(word).lower() in stop_list

Ensuite, nous créons une fonction preprocess pour:
- filter les mots vides et les symboles de ponctuation
- normaliser les mots (mettre tout en minuscules)
- supprimer tout text qui n'est pas alphanumérique (à l'aide d'expressions régulières en important le module 're')

In [ ]:
import re

# Fonction pour le pré-traitement des textes
def preprocess(list_text):
    filtered_sent=[]
    for text in list_text:
        text = str(text).strip().lower()
        text = re.sub(r'[^A-Za-z0-9]+',' ',text)
        doc = nlp(text)
        # filtering stop words
        token_list = []
        for word in doc:
            if not is_stop_word(word,en_stopwords) and word.is_punct==False:
                token_list.append(word.text)
        filtered_sent.append(' '.join(token_list))
    #print("Filtered Sentence:",filtered_sent)
    return filtered_sent

In [ ]:
df["review"]=preprocess(df["review"])
df.head(5)

,review,sentiment
0,reviewers mentioned watching 1 oz episode ll h...,1
1,wonderful little production filming technique ...,1
2,thought wonderful way spend time hot summer we...,1
3,basically s family little boy jake thinks s zo...,0
4,petter mattei s love time money visually stunn...,1


## Séparer la base de données en ensemble d'apprentissage et de test

Nous determinons la taille de l'ensemble d'apprentissage (par exemple: 80%). C'est notre train_rate.

Nous prenons le nombre total de documents (ici: 50000). C'est la valeur retournée par df.shape[0]

Et on fixe la taille du training à 80% du nbre total des données.

In [ ]:
import numpy as np

train_rate=0.80
dataset_size=df.shape[0]
train_size=int(train_rate*dataset_size)
print(f"Taille de la base de données {dataset_size} de l'ensemble d'apprentissage {train_size} et de test {dataset_size-train_size}")

Taille de la base de données 50000 de l'ensemble d'apprentissage 40000 et de test 10000


Ensuite, on tire au hasard les indices du training en utilisant la fonction **random.choice()** de numpy.
On met replace=False pour éviter de selectionner des éléments qui se répétent. Enfin, on trie les indices tirés.

In [ ]:
train_indices=np.random.choice(dataset_size,train_size, replace=False)
train_indices=sorted(train_indices)

Puis, on collecte les indices qui ne sont pas dans le train et on les met systématiquement dans le test.

In [ ]:
train_indices_dict = {n:n for n in train_indices}
test_indices=[]
for n in range(dataset_size):
    try:
        train_indices_dict[n]
    except:
        test_indices.append(n)

Avec la fonction iloc, nous pouvons extraire maintenant les indices de train et ceux de test.

In [ ]:
df_train=df.iloc[train_indices]
df_test=df.iloc[test_indices]

print(f"Taille de l'ensemble de train {df_train.shape[0]}")
print(f"Taille de l'ensemble de test {df_test.shape[0]}")

Taille de l'ensemble de train 40000
Taille de l'ensemble de test 10000


On affiche les premiers documents du train.

In [ ]:
df_train.head()

,review,sentiment
0,reviewers mentioned watching 1 oz episode ll h...,1
1,wonderful little production filming technique ...,1
3,basically s family little boy jake thinks s zo...,0
4,petter mattei s love time money visually stunn...,1
5,probably time favorite movie story selflessnes...,1


On affiche également les premiers documents du test.

In [ ]:
df_test.head()

,review,sentiment
2,thought wonderful way spend time hot summer we...,1
7,amazing fresh innovative idea 70 s aired 7 8 y...,0
14,fantastic movie prisoners famous actors george...,1
15,kind drawn erotic scenes realize amateurish un...,0
21,terrible misfortune having view b movie s enti...,0


Une autre possibilité serait d'utiliser la fonction train_test_split de sklearn.model_selection pour séparer les données des ensembles d'apprentissage et de test.


In [ ]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(df.review,df.sentiment,test_size=0.2)

# Extraction de features

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

In [ ]:
tfidf_vect = TfidfVectorizer()

In [ ]:
x_train = tfidf_vect.fit_transform(x_train)

In [ ]:
terms = tfidf_vect.get_feature_names()

In [ ]:
x_train.shape

(40000, 94000)

Nous appliquons la même transformation à l'ensemble de test. Attention, ici on ne doit pas appliquer la fonction fit_transform() dans la mesure où les statistiques ont été collectées sur le training et non sur le test. On veut juste appliquer les mêmes calculs au test.

In [ ]:
x_test = tfidf_vect.transform(x_test)

In [ ]:
terms = tfidf_vect.get_feature_names()
df_test = pd.DataFrame(x_test.toarray(), columns=terms)
df_test

,00,000,0000000000001,00000001,00001,00015,000s,001,003830,006,...,zzzz,zzzzip,zzzzz,zzzzzzzz,zzzzzzzzz,zzzzzzzzzzzzpop,zzzzzzzzzzzzz,zzzzzzzzzzzzzzzzzz,zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz,zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9996,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Classification de reviews

## Classification basée sur Naive Bayes

On importe leclassificateur multinomial Naive Bayes (MultinomialNB). Celui-ci convient à la classification avec des caractéristiques discrètes (par exemple, le nombre de mots pour la classification de textes).

In [ ]:
from sklearn.naive_bayes import MultinomialNB

Maintenant, nous créons un modèle de Naive Bayes multinomial appelée MultinomialNB().

In [ ]:
nb_clf = MultinomialNB()
nb_clf.fit(x_train, y_train)

MultinomialNB()

Après l'apprentissage, nous pouvons prédire le test maintenant.

In [ ]:
y_pred=nb_clf.predict(x_test)

Nous comparons les classes prédites par le système aux classes réelles. On afficher le rapport d'evaluation avec les valeurs de précision, rappel et f-score.

In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.86      0.87      0.87      4955
           1       0.87      0.86      0.87      5045

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



Nous créons deux phrases test: une qui contient des sentiments negatifs et une qui contient des sentiments positifs.

In [ ]:
phrase_test1 = 'This movie is horrible. I hate these stupid actors.'

In [ ]:
y_pred1 = nb_clf.predict(tfidf_vect.transform([phrase_test1]))
y_pred1

array([0])

In [ ]:
phrase_test2 = 'This movie is great. I like these funny actors.'

In [ ]:
y_pred2 = nb_clf.predict(tfidf_vect.transform([phrase_test2]))
y_pred2

array([1])